# Challenge 4: Advanced Composition (Bonus)

You've built the core system — now push it further. Pick **one or more** of these advanced patterns.

| Option | Difficulty | What You'll Learn | Time |
|--------|-----------|-------------------|------|
| **A: Workflow-as-Agent** | ⭐⭐ | Your entire workflow becomes a single callable agent | 10 min |
| **B: OpenTelemetry** | ⭐ | Distributed tracing for every agent/tool call | 10 min |
| **C: Sub-Workflow** | ⭐⭐⭐ | Nested workflows as reusable graph nodes | 15 min |
| **D: Parallel Fan-Out** | ⭐⭐⭐ | Investigate multiple services concurrently | 15 min |

## Setup

In [ ]:
import os
import sys
import json
from typing import Any, Literal
from dataclasses import dataclass

sys.path.insert(0, "..")
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from agent_framework import (
    Agent, AgentExecutor, AgentExecutorRequest, AgentExecutorResponse,
    Case, Default, Message, WorkflowBuilder, WorkflowContext,
    WorkflowRunState, executor, handler, response_handler, tool,
)
from agent_framework.foundry import FoundryChatClient
from agent_framework.openai import OpenAIChatOptions
from azure.identity import AzureCliCredential

from tools.mock_infra import (
    check_alert_history, get_runbook,
    get_metrics, get_logs, check_dependencies,
    get_health_status, run_smoke_test,
    restart_pod, scale_service, flush_cache, toggle_feature_flag,
    post_to_slack, create_incident_ticket,
)

load_dotenv("../.env")

with open("../data/incidents.json") as f:
    incidents = json.load(f)

client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["FOUNDRY_MODEL"],
    credential=AzureCliCredential(),
)

print("✅ Setup ready")

## Rebuild the Challenge 2 Workflow

We need the working workflow from Challenge 2 as a base for all options.
Run this cell — it rebuilds the triage → diagnostics pipeline.

In [ ]:
# Models
class TriageResult(BaseModel):
    severity: Literal["critical", "high", "medium", "low"]
    is_recurring: bool
    auto_remediation_allowed: bool
    root_cause_hypothesis: str
    recommended_action: str
    escalation_threshold_minutes: int

class DiagnosticsResult(BaseModel):
    root_cause: str
    evidence: list[str]
    affected_components: list[str]
    confidence: float = Field(ge=0.0, le=1.0)
    recommended_fix: str
    requires_restart: bool

# Agents
triage_agent = Agent(
    client, id="triage-agent", name="TriageAgent",
    instructions="You are a Triage Agent. Call check_alert_history and get_runbook, then classify severity.",
    tools=[check_alert_history, get_runbook],
    default_options=OpenAIChatOptions(response_format=TriageResult),
)
diagnostics_agent = Agent(
    client, id="diagnostics-agent", name="DiagnosticsAgent",
    instructions="You are a Diagnostics Agent. Call get_metrics, get_logs, check_dependencies. Synthesize root cause.",
    tools=[get_metrics, get_logs, check_dependencies],
    default_options=OpenAIChatOptions(response_format=DiagnosticsResult),
)

# Executors
@dataclass
class RoutingDecision:
    severity: str
    service: str

@executor(id="ingest_alert")
async def ingest_alert(alert_json: str, ctx: WorkflowContext) -> None:
    alert = json.loads(alert_json)
    ctx.set_state("alert", alert)
    ctx.set_state("service", alert["service"])
    msg = Message("user", contents=[
        f"Alert: {alert['title']}\nService: {alert['service']}\n"
        f"Type: {alert['incident_type']}\nDescription: {alert['description']}"
    ])
    await ctx.send_message(AgentExecutorRequest(messages=[msg], should_respond=True))

@executor(id="parse_triage")
async def parse_triage(response: AgentExecutorResponse, ctx: WorkflowContext) -> None:
    triage = TriageResult.model_validate_json(response.agent_response.text)
    ctx.set_state("triage_result", triage)
    await ctx.send_message(RoutingDecision(severity=triage.severity, service=ctx.get_state("service")))

def needs_diagnostics(msg: Any) -> bool:
    return isinstance(msg, RoutingDecision) and msg.severity in ("critical", "high")

@executor(id="to_diagnostics")
async def to_diagnostics(routing: RoutingDecision, ctx: WorkflowContext) -> None:
    triage = ctx.get_state("triage_result")
    msg = Message("user", contents=[
        f"Investigate: {ctx.get_state('service')}\nHypothesis: {triage.root_cause_hypothesis}"
    ])
    await ctx.send_message(AgentExecutorRequest(messages=[msg], should_respond=True))

@executor(id="monitor_only")
async def monitor_only(routing: RoutingDecision, ctx: WorkflowContext) -> None:
    alert = ctx.get_state("alert")
    await ctx.yield_output(f"📋 LOW: {alert['title']} — monitoring only.")

@executor(id="comms")
async def comms(response: AgentExecutorResponse, ctx: WorkflowContext) -> None:
    diag = DiagnosticsResult.model_validate_json(response.agent_response.text)
    triage = ctx.get_state("triage_result")
    ctx.set_state("diagnostics_result", diag)
    report = (
        f"🚨 INCIDENT RESPONSE REPORT\n{'='*40}\n"
        f"Service: {ctx.get_state('service')}\n"
        f"Severity: {triage.severity.upper()}\n"
        f"Root Cause: {diag.root_cause}\n"
        f"Confidence: {diag.confidence:.0%}\n"
        f"Fix: {diag.recommended_fix}\n"
    )
    await ctx.yield_output(report)

# Build
triage_exec = AgentExecutor(triage_agent)
diag_exec = AgentExecutor(diagnostics_agent)
base_workflow = (
    WorkflowBuilder(start_executor=ingest_alert)
    .add_edge(ingest_alert, triage_exec)
    .add_edge(triage_exec, parse_triage)
    .add_switch_case_edge_group(parse_triage, [
        Case(condition=needs_diagnostics, target=to_diagnostics),
        Default(target=monitor_only),
    ])
    .add_edge(to_diagnostics, diag_exec)
    .add_edge(diag_exec, comms)
    .build()
)
print("✅ Base workflow rebuilt from Challenge 2")

---
# Option A: Workflow-as-Agent ⭐⭐

Your entire incident response workflow becomes a **single callable agent**.
A supervisor can invoke it alongside other agents without knowing it's a full graph internally.

```
┌──────────────────────────────┐
│     Supervisor Agent         │
│         │                    │
│   ┌─────▼──────┐  ┌──────┐  │
│   │ Incident   │  │ Other│  │
│   │ Workflow   │  │ Agent│  │
│   │ (as agent) │  │      │  │
│   └────────────┘  └──────┘  │
└──────────────────────────────┘
```

<div style="border: 1px solid #e94560; border-left: 4px solid #e94560; padding: 16px 20px; border-radius: 6px; background-color: rgba(233, 69, 96, 0.08);">

**✍️ Your Turn** — Wrap the workflow as an agent and call it

Place your cursor in the cell below and press `Ctrl+I` to let Copilot generate the code from the comments.

</div>

In [ ]:
# Wrap base_workflow as an agent using base_workflow.as_agent():
# 1. incident_agent = base_workflow.as_agent(name="IncidentResponder", instructions="Handle production incidents.")
# 2. Run it like a normal agent: response = await incident_agent.run(json.dumps(incidents[0]))
# 3. Print the response text
# The caller doesn't know it's a full workflow internally!

In [ ]:
# Validate
assert response is not None, "Should get a response"
print("✅ Option A: Workflow-as-Agent works!")
print(f"   The caller sees a simple agent — the full pipeline ran internally.")

---
# Option B: OpenTelemetry Observability ⭐

MAF instruments all agent calls and tool invocations with OpenTelemetry spans automatically.
Just configure an exporter and get full distributed tracing.

```
Trace: handle_incident (14.2s)
├── Span: ingest_alert (2ms)
├── Span: triage_agent (3.1s)
│   ├── Span: tool:check_alert_history (120ms)
│   └── Span: tool:get_runbook (95ms)
├── Span: diagnostics_agent (4.8s)
│   ├── Span: tool:get_metrics (340ms)
│   └── Span: tool:get_logs (280ms)
└── Span: comms (1.2s)
```

<div style="border: 1px solid #e94560; border-left: 4px solid #e94560; padding: 16px 20px; border-radius: 6px; background-color: rgba(233, 69, 96, 0.08);">

**✍️ Your Turn** — Add OpenTelemetry tracing to the workflow

Place your cursor in the cell below and press `Ctrl+I` to let Copilot generate the code from the comments.

</div>

In [ ]:
# Set up OpenTelemetry tracing with ConsoleSpanExporter:
# 1. from opentelemetry import trace
# 2. from opentelemetry.sdk.trace import TracerProvider
# 3. from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor
# 4. provider = TracerProvider()
# 5. provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
# 6. trace.set_tracer_provider(provider)
# 7. Run the base_workflow on incidents[0]: events = await base_workflow.run(json.dumps(incidents[0]))
# 8. Print events.get_outputs() — spans will appear in console automatically

In [ ]:
print("✅ Option B: OpenTelemetry tracing active!")
print("   Check the console output above — every agent call and tool invocation has a span.")
print("   In production, export to Azure Monitor or Jaeger for dashboards.")

---
# Option C: Sub-Workflow Composition ⭐⭐⭐

Extract remediation into a **reusable sub-workflow** — plan → approve → execute.
Then plug it into the main graph as a single node.

```
Main:    ingest → triage → diagnostics → [remediation_sub] → comms
Sub:     plan → approve(HITL) → execute → verify
```

<div style="border: 1px solid #e94560; border-left: 4px solid #e94560; padding: 16px 20px; border-radius: 6px; background-color: rgba(233, 69, 96, 0.08);">

**✍️ Your Turn** — Build a remediation sub-workflow

Place your cursor in the cell below and press `Ctrl+I` to let Copilot generate the code from the comments.

</div>

In [ ]:
# Build a remediation sub-workflow with HITL approval:
# 1. Import Executor, handler, response_handler from agent_framework
# 2. Create a PlanExecutor(@executor) that receives diagnostics text and yields a remediation plan string
# 3. Create an ApprovalGate(Executor class) with:
#    - @handler that calls await ctx.request_info(f"Approve: {message}?", str)
#    - @response_handler that forwards the response via ctx.send_message()
# 4. Create an ExecuteAction(@executor) that checks if response contains "yes":
#    - If yes: call restart_pod("payment-api", "pod-3") and yield_output the result
#    - If no: yield_output "Aborted"
# 5. Wire: WorkflowBuilder(start_executor=plan).add_edge(plan, gate).add_edge(gate, execute).build()
# 6. Test it: result = await sub_workflow.run("restart payment-api pod-3")

---
# Option D: Parallel Fan-Out ⭐⭐⭐

Incident #1 (payment-api) cascades to incident #2 (order-service).
Fan-out to diagnose **both services concurrently**.

```
                    ┌── diagnostics_payment ──┐
detect_affected ───→│                         ├──→ aggregate → remediate
                    └── diagnostics_orders  ──┘
                       (runs in parallel!)
```

<div style="border: 1px solid #e94560; border-left: 4px solid #e94560; padding: 16px 20px; border-radius: 6px; background-color: rgba(233, 69, 96, 0.08);">

**✍️ Your Turn** — Build a parallel fan-out workflow

Place your cursor in the cell below and press `Ctrl+I` to let Copilot generate the code from the comments.

</div>

In [ ]:
# Build a parallel fan-out workflow that diagnoses multiple services concurrently:
# 1. Create @executor detect_affected that reads incidents[0] and emits messages
#    for both "payment-api" and "order-service" (check_dependencies shows the cascade)
# 2. Create two diagnostics executors — one for payment-api, one for order-service
#    Each wraps a diagnostics Agent with get_metrics, get_logs, check_dependencies
# 3. Create @executor aggregate_results that collects both results and yields
#    a combined report showing root cause vs cascading symptom
# 4. Wire with parallel edges:
#    detect → diag_payment (condition: affects payment-api)
#    detect → diag_orders  (condition: affects order-service)
#    diag_payment → aggregate
#    diag_orders  → aggregate
# 5. Run and print the aggregated output

---
## 🎉 Workshop Complete!

### What You Built

```
Challenge 1: Structured Agents    → Typed contracts between agents
Challenge 2: Workflow Graphs       → DAG orchestration with conditional routing
Challenge 3: Human-in-the-Loop    → Safe operations with approval gates
Challenge 4: Advanced Composition  → Production patterns (tracing, parallelism, composition)
```

### Take Home

The `maf-lab` repo is yours. Fork it, extend it, adapt it to your own use cases.
The patterns here work for any domain — not just incident response.

### Resources

- [MAF GitHub](https://github.com/microsoft/agent-framework)
- [MAF Workflow Samples](https://github.com/microsoft/agent-framework/tree/main/python/samples/03-workflows)
- [Azure AI Foundry](https://ai.azure.com)